# Exercises for Section 3.2 Confidence intervals

This notebook contains the solutions to the exercises
from [Section 3.2 Confidence intervals]()
in the **No Bullshit Guide to Statistics**.

### Notebooks setup

In [1]:
# Ensure required Python modules are installed
%pip install --quiet numpy scipy seaborn pandas ministats

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Figures setup
plt.clf()  # needed otherwise `sns.set_theme` doesn't work
sns.set_theme(
    context="paper",
    style="whitegrid",
    palette="colorblind",
)
%config InlineBackend.figure_format = "retina"

<Figure size 640x480 with 0 Axes>

In [4]:
# Simplified int and float __repr__
np.set_printoptions(legacy='1.25')

In [5]:
# Download datasets/ directory if necessary
from ministats import ensure_datasets
ensure_datasets()

datasets/ directory already exists.


$\def\stderr#1{\mathbf{se}_{#1}}$
$\def\stderrhat#1{\hat{\mathbf{se}}_{#1}}$
$\newcommand{\Mean}{\textbf{Mean}}$
$\newcommand{\Var}{\textbf{Var}}$
$\newcommand{\Std}{\textbf{Std}}$
$\newcommand{\Freq}{\textbf{Freq}}$
$\newcommand{\RelFreq}{\textbf{RelFreq}}$
$\newcommand{\DMeans}{\textbf{DMeans}}$
$\newcommand{\Prop}{\textbf{Prop}}$
$\newcommand{\DProps}{\textbf{DProps}}$
$$
\newcommand{\CI}[1]{\textbf{CI}_{#1}}
\newcommand{\CIL}[1]{\textbf{L}_{#1}}
\newcommand{\CIU}[1]{\textbf{U}_{#1}}
\newcommand{\ci}[1]{\textbf{ci}_{#1}}
\newcommand{\cil}[1]{\textbf{l}_{#1}}
\newcommand{\ciu}[1]{\textbf{u}_{#1}}
$$
(this cell contains the macro definitions like $\stderr{\overline{\mathbf{x}}}$, $\stderrhat{}$, $\Mean$, ...)

## Exercises

### E3.17

<!-- {exercise:ci90-kombucha-pop-mean-from-batch-04} -->

Construct a 90% confidence interval for the population mean
based on the sample from Batch 04 of the kombucha dataset.

**a)** Use an analytical approximation based on Student's $t$-distribution.

**b)** Use the percentile bootstrap method.

In [6]:
# Load the kombucha dataset
kombucha = pd.read_csv("datasets/kombucha.csv")

# Select the volume measurements from Batch 04
ksample04 = kombucha[kombucha["batch"]==4]["volume"]

# Compute the sample size
n04 = len(ksample04)
n04

40

#### a) Analytical approximation

In [8]:
# Compute the estimated standard error of the mean for samples of size n=40
kbar04 = np.mean(ksample04)
std04 = ksample04.std()
seKbar04 = std04 / np.sqrt(n04)

# Find the 0.05 and 0.95 quantiles of the standard t-dist. with ν=39
from scipy.stats import t as tdist
rvT = tdist(df=n04-1)
t_l, t_u = rvT.ppf(0.05), rvT.ppf(0.95)

# Construct the 90% confidence interval for the population mean
[kbar04 + t_l*seKbar04, kbar04 + t_u*seKbar04]

[1001.7416639437092, 1005.9253360562907]

#### b) Bootstrap estimate

In [10]:
# Set the random seed to a particular value to get a reproducible result
np.random.seed(42)

# Generate the bootstrap distribution of the mean
from ministats import gen_boot_dist
kbars04_boot = gen_boot_dist(ksample04, estfunc=np.mean)

# Construct the 90% bootstrap confidence interval for the population mean
[np.quantile(kbars04_boot,0.05), np.quantile(kbars04_boot,0.95)]

[1001.8019875, 1005.8771125000001]

The results we obtained in **a)** and **b)** are very similar.

### E3.18

<!-- {exercise:ci90-kombucha-var-from-batch-05} -->

Construct a 90% confidence interval for the population variance
based on the data from Batch 05 of the kombucha dataset.

**a)** Use an analytical approximation based on the $\chi^2$-distribution.

**b)** Use the percentile bootstrap method.


In [11]:
# Define the variance estimator function
def var(sample):
    xbar = sum(sample) / len(sample)
    sumsqdevs = sum([(xi-xbar)**2 for xi in sample])
    return sumsqdevs / (len(sample)-1)

In [12]:
# Load the Batch 05 data from the kombucha dataset
kombucha = pd.read_csv("datasets/kombucha.csv")
ksample05 = kombucha[kombucha["batch"]==5]["volume"]

#### a) Analytical approximation

In [14]:
# Compute the properties of the sample `ksample05`
n05 = len(ksample05)
var05 = var(ksample05)

# Find the 0.05 and 0.95 quantiles of the standard chi-square dist.
from scipy.stats import chi2
q_l = chi2(df=n05-1).ppf(0.05)
q_u = chi2(df=n05-1).ppf(0.95)

# Construct the 90% confidence interval for the population variance
[(n05-1)*var05/q_u,  (n05-1)*var05/q_l]

[30.692160944915106, 65.1844385881649]

#### b) Bootstrap estimate

In [16]:
# Set the random seed to a particular value to get a reproducible result
np.random.seed(43)

# Generate the bootstrap distribution of the variance
from ministats import gen_boot_dist
kvars05_boot = gen_boot_dist(ksample05, estfunc=var)

# Construct the 90% bootstrap confidence interval for the population variance
[np.quantile(kvars05_boot,0.05), np.quantile(kvars05_boot,0.95)]

[20.34142641025637, 70.91556086217975]

The lower bound of the bootstrap confidence interval
is smaller than the one we obtained using the analytical approximation,
but overall,
the two intervals are in the same ballpark.

### E3.21

<!-- {exercise:ci95-rats} -->

You have obtained a sample $\mathbf{x}$ of size $n=64$.
The sample mean is $\overline{\mathbf{x}} = 89.60$
with standard deviation $s_{\mathbf{x}} =12.96$.

**a)** Construct a 95% confidence interval for the population mean $\mu$
using an approximation based on the normal model.

**b)** Construct a 95% confidence interval using Student's $t$-distribution.

**c)** Compare your answers to parts **a)** and **b)**
and comment on the need for Student's $t$-distribution in this scenario.

In [29]:
# Descriptive statistics of the sample
n = 64
xbar = 89.60
sx = 12.96

#### a) Using normal approximation

In [31]:
# Compute the estimated standard error of the mean for samples of size n
sehat = sx / np.sqrt(n)
sehat

1.62

In [33]:
# Find the 0.025 and 0.975 quantiles of the standard normal
from scipy.stats import norm
z_l = norm.ppf(0.025)
z_u = norm.ppf(0.975)

# Construct a 95% CI for the population mean
[xbar + z_l*sehat, xbar + z_u*sehat]

[86.42485834504511, 92.77514165495488]

In [34]:
# ALT. using normal with custom `loc` and `scale` params
rvNXbar = norm(loc=xbar, scale=sehat)
[rvNXbar.ppf(0.025), rvNXbar.ppf(0.975)]

[86.42485834504511, 92.77514165495488]

#### b) Using Student's $t$-distribution

In [36]:
# Find the 0.025 and 0.975 quantiles of the standard t-dist.
from scipy.stats import t as tdist
t_l = tdist(df=n-1).ppf(0.025)
t_u = tdist(df=n-1).ppf(0.975)

# Construct a 95% CI for the population mean
[xbar + t_l*sehat, xbar + t_u*sehat]

[86.3626883211164, 92.8373116788836]

In [37]:
# ALT. using t-distribution with custom `loc` and `scale` params
rvTXbar = tdist(n-1, loc=xbar, scale=sehat)
[rvTXbar.ppf(0.025), rvTXbar.ppf(0.975)]

[86.3626883211164, 92.8373116788836]

### E3.22

<!-- {exercise:ci-90-theta-explain-in-words} -->

You calculate the 90% confidence interval
$\ci{\theta,0.9} = [\cil{\theta}, \ciu{\theta}]$
for the population parameter $\theta$.
Explain in your own words what $\ci{\theta,0.9}$ means.
What type of guarantee does $\ci{\theta,0.9}$ provide?

Hint: Note the question is asking about $\ci{\theta,0.9}$ and not $\CI{\theta,0.9}$.

The 90% confidence interval $\ci{\theta,0.9} = [\cil{\theta}, \ciu{\theta}]$
gives us a range of plausible numbers that we believe contains the true population parameter $\theta$.
In other words,
we believe that $\theta \in \ci{\theta,0.9}$.
We can't give any specific guarantee for the specific confidence interval $\ci{\theta,0.9}$
we obtained in this analysis,
but we know that the *procedure* we used to construct $\ci{\theta,0.9}$
produces intervals that contain $\theta$ in $90\%$ of repeated samples.